# 🎬 Movie Matrix AI — Kapsamlı Analiz Notebook'u

**TMDB 5000 Dataset | EDA | Veri Temizleme | ML Modelleri | Grafiksel Analizler**

---

Bu notebook aşağıdaki bölümleri içermektedir:

| # | Bölüm |
|---|-------|
| 1 | 📦 Kurulum & Kütüphaneler |
| 2 | 📁 Veri Yükleme |
| 3 | 🧹 Veri Temizleme |
| 4 | 📊 Keşifsel Veri Analizi (EDA) |
| 5 | 🔥 Isı Haritası (Korelasyon) |
| 6 | 🤖 Random Forest Analizi |
| 7 | 📈 Lineer Regresyon |
| 8 | 🎯 Model Doğruluk Metrikleri |
| 9 | 🎬 Film Test Et (Input) |

## 📦 1. Kurulum & Kütüphaneler

In [ ]:
# Gerekli kütüphaneleri kur
!pip install scikit-learn pandas numpy matplotlib seaborn plotly -q
print('✅ Kurulum tamamlandı!')

In [ ]:
import ast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, accuracy_score
)

warnings.filterwarnings('ignore')

# Matplotlib stil ayarları
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('✅ Kütüphaneler başarıyla import edildi!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : yüklü')

## 📁 2. Veri Yükleme

> **Google Drive'dan yüklemek için** `USE_DRIVE = True` yap ve `PROJECT_PATH`'i güncelle.  
> **Manuel yüklemek için** `USE_DRIVE = False` yap ve CSV dosyalarını sol panelden yükle.

In [ ]:
import os

USE_DRIVE = True  # False → Manuel yükleme

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/movie_matrix_ai'  # ← Kendi yolunu yaz
    os.chdir(PROJECT_PATH)
    print(f'📂 Çalışma dizini: {os.getcwd()}')
else:
    from google.colab import files
    print('⬆️  Aşağıdaki dosyaları yükle:')
    print('   • tmdb_5000_movies.csv')
    print('   • tmdb_5000_credits.csv')
    uploaded = files.upload()

In [ ]:
print('📂 CSV dosyaları yükleniyor...')
movies_raw = pd.read_csv('tmdb_5000_movies.csv')
credits_raw = pd.read_csv('tmdb_5000_credits.csv')

print(f'✅ Movies  : {movies_raw.shape[0]:,} satır × {movies_raw.shape[1]} sütun')
print(f'✅ Credits : {credits_raw.shape[0]:,} satır × {credits_raw.shape[1]} sütun')
print('\n--- Movies sütunları ---')
print(list(movies_raw.columns))
print('\n--- Credits sütunları ---')
print(list(credits_raw.columns))

## 🧹 3. Veri Temizleme

Veri temizleme adımları:
1. Merge işlemi
2. Eksik değer analizi
3. Sıfır bütçe/gelir kayıtlarını temizle
4. JSON sütunlarını parse et
5. Feature engineering (ROI, kâr, dönem, vb.)

In [ ]:
# ── 3.1 Birleştirme ──────────────────────────────────────────────────────────
df_merged = movies_raw.merge(credits_raw, on='title')
print(f'✅ Merged  : {df_merged.shape[0]:,} satır')

# ── 3.2 Eksik değer analizi ──────────────────────────────────────────────────
print('\n📊 Eksik Değer Analizi (İlk Merge Sonrası):')
missing = df_merged.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    missing_df = pd.DataFrame({
        'Sütun': missing.index,
        'Eksik Sayı': missing.values,
        'Eksik %': (missing.values / len(df_merged) * 100).round(2)
    })
    print(missing_df.to_string(index=False))
else:
    print('Hiç eksik değer yok!')

In [ ]:
# ── 3.3 Eksik değerleri görselleştir ─────────────────────────────────────────
missing_all = df_merged.isnull().sum()
missing_all = missing_all[missing_all > 0].sort_values(ascending=True)

if len(missing_all) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(missing_all.index, missing_all.values, color='#e74c3c', alpha=0.8)
    ax.set_xlabel('Eksik Değer Sayısı')
    ax.set_title('📊 Merge Sonrası Eksik Değer Dağılımı', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, missing_all.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('Görselleştirilecek eksik değer yok.')

In [ ]:
# ── 3.4 Sıfır bütçe/gelir filtresi ──────────────────────────────────────────
n_before = len(df_merged)
df_clean = df_merged[(df_merged['budget'] > 0) & (df_merged['revenue'] > 0)].copy()
n_after = len(df_clean)
print(f'🗑️  Sıfır bütçe/gelir kaldırıldı: {n_before - n_after:,} kayıt')
print(f'✅ Temiz veri: {n_after:,} kayıt\n')

# ── 3.5 JSON sütunlarını parse et ────────────────────────────────────────────
def parse_json_col(obj):
    try: return [i['name'] for i in ast.literal_eval(obj)]
    except: return []

def parse_crew_role(obj, job_filter):
    try: return [i['name'] for i in ast.literal_eval(obj) if i.get('job') == job_filter]
    except: return []

def parse_cast(obj, limit=5):
    try: return [i['name'] for i in ast.literal_eval(obj)][:limit]
    except: return []

print('⚙️  JSON sütunları parse ediliyor...')
df_clean['genres_list']    = df_clean['genres'].apply(parse_json_col)
df_clean['keywords_list']  = df_clean['keywords'].apply(parse_json_col)
df_clean['companies_list'] = df_clean['production_companies'].apply(parse_json_col)
df_clean['cast_list']      = df_clean['cast'].apply(lambda x: parse_cast(x, 5))
df_clean['director_list']  = df_clean['crew'].apply(lambda x: parse_crew_role(x, 'Director'))
df_clean['director']       = df_clean['director_list'].apply(lambda x: x[0] if x else 'Unknown')
print('✅ JSON parse tamamlandı!')

# ── 3.6 Feature Engineering ──────────────────────────────────────────────────
print('⚙️  Feature engineering...')
df_clean['roi']           = df_clean['revenue'] / df_clean['budget']
df_clean['profit']        = df_clean['revenue'] - df_clean['budget']
df_clean['profit_m']      = df_clean['profit'] / 1e6
df_clean['budget_m']      = df_clean['budget'] / 1e6
df_clean['revenue_m']     = df_clean['revenue'] / 1e6
df_clean['success_class'] = df_clean['roi'].apply(lambda x: 2 if x > 2 else (1 if x > 1 else 0))
df_clean['success_label'] = df_clean['success_class'].map({2: 'Hit', 1: 'Orta', 0: 'Riskli'})
df_clean['release_year']  = pd.to_datetime(df_clean['release_date'], errors='coerce').dt.year
df_clean['decade']        = (df_clean['release_year'] // 10 * 10).astype('Int64').astype(str) + 's'
df_clean['genre_count']   = df_clean['genres_list'].apply(len)

# Yönetmen ortalama ROI
dir_roi = df_clean[df_clean['roi'].between(0, 1000)].groupby('director')['roi'].mean()
df_clean['director_avg_roi'] = df_clean['director'].map(dir_roi).fillna(dir_roi.median())

# Oyuncu gişe gücü
cast_rev = df_clean.explode('cast_list').groupby('cast_list')['revenue'].mean()
df_clean['cast_power'] = df_clean['cast_list'].apply(
    lambda cl: np.mean([cast_rev.get(c, 0) for c in cl]) if cl else 0
)

# Aşırı değerleri maskele (görselleştirme için)
df_clean = df_clean[df_clean['roi'] < 200].reset_index(drop=True)

print(f'✅ Feature engineering tamamlandı!')
print(f'   Toplam kullanılabilir film: {len(df_clean):,}')
print(f'   Hit filmler  : {(df_clean["success_class"]==2).sum():,}')
print(f'   Orta filmler : {(df_clean["success_class"]==1).sum():,}')
print(f'   Riskli filmler: {(df_clean["success_class"]==0).sum():,}')

# Ana DataFrame'i kaydet
df = df_clean.copy()

In [ ]:
# ── 3.7 Temizleme özeti tablosu ───────────────────────────────────────────────
print('='*55)
print('  VERİ TEMİZLEME ÖZETİ')
print('='*55)
print(f'  Ham movies kayıt sayısı   : {len(movies_raw):>6,}')
print(f'  Merge sonrası             : {len(df_merged):>6,}')
print(f'  Sıfır bütçe/gelir sonrası : {n_after:>6,}')
print(f'  Aşırı değer temizleme son : {len(df):>6,}')
print(f'  Elde edilen özellik sayısı: {len(df.columns):>6}')
print('='*55)

## 📊 4. Keşifsel Veri Analizi (EDA)

Veri setinin genel yapısını ve içindeki örüntüleri keşfediyoruz.

In [ ]:
# ── 4.1 Temel İstatistikler ───────────────────────────────────────────────────
stats_cols = ['budget_m', 'revenue_m', 'roi', 'vote_average', 'popularity', 'runtime']
stats = df[stats_cols].describe().T
stats.columns = ['N', 'Ort.', 'Std', 'Min', 'Q1', 'Medyan', 'Q3', 'Max']
stats = stats.round(2)
print('📋 Sayısal Sütun İstatistikleri:')
print(stats.to_string())

In [ ]:
# ── 4.2 Gişe ve Bütçe Dağılımı ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bütçe histogramı
axes[0].hist(df['budget_m'], bins=50, color='#3498db', alpha=0.8, edgecolor='white')
axes[0].set_title('💰 Bütçe Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Bütçe (Milyon $)')
axes[0].set_ylabel('Film Sayısı')
axes[0].axvline(df['budget_m'].median(), color='red', linestyle='--', label=f'Medyan: ${df["budget_m"].median():.0f}M')
axes[0].legend()

# Gelir histogramı
axes[1].hist(df['revenue_m'], bins=50, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[1].set_title('💵 Gişe Geliri Dağılımı', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gişe Geliri (Milyon $)')
axes[1].set_ylabel('Film Sayısı')
axes[1].axvline(df['revenue_m'].median(), color='red', linestyle='--', label=f'Medyan: ${df["revenue_m"].median():.0f}M')
axes[1].legend()

plt.suptitle('📊 Bütçe ve Gişe Geliri Dağılımı', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3 ROI Dağılımı ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

roi_plot = df[df['roi'] < 20]  # aşırı değerleri görselleştirmeden hariç tut

axes[0].hist(roi_plot['roi'], bins=60, color='#e67e22', alpha=0.85, edgecolor='white')
axes[0].axvline(1, color='red', linestyle='--', linewidth=2, label='ROI=1 (Başabaş)')
axes[0].axvline(2, color='green', linestyle='--', linewidth=2, label='ROI=2 (Hit Sınırı)')
axes[0].set_title('📈 ROI Dağılımı (ROI < 20)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('ROI (Revenue / Budget)')
axes[0].set_ylabel('Film Sayısı')
axes[0].legend()

# Başarı sınıfı pasta grafiği
success_counts = df['success_label'].value_counts()
colors = ['#e74c3c', '#f39c12', '#27ae60']
wedges, texts, autotexts = axes[1].pie(
    success_counts.values,
    labels=success_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=(0.05, 0.05, 0.05)
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('🎯 Başarı Sınıfı Dağılımı', fontsize=14, fontweight='bold')

plt.suptitle('📊 ROI ve Başarı Analizi', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\n📊 Başarı Sınıfı Dağılımı:')
for label, count in success_counts.items():
    print(f'  {label:8s}: {count:4,} film ({count/len(df)*100:.1f}%)')

In [ ]:
# ── 4.4 En Popüler Türler (Bar Chart) ─────────────────────────────────────────
from collections import Counter

all_genres = [g for genres in df['genres_list'] for g in genres]
genre_counts = Counter(all_genres).most_common(15)
genre_df = pd.DataFrame(genre_counts, columns=['Tür', 'Film Sayısı'])

fig, ax = plt.subplots(figsize=(13, 7))
palette = sns.color_palette('husl', len(genre_df))
bars = ax.barh(genre_df['Tür'][::-1], genre_df['Film Sayısı'][::-1], color=palette[::-1], alpha=0.85)

for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=10)

ax.set_xlabel('Film Sayısı', fontsize=12)
ax.set_title('🎭 En Popüler Film Türleri (Top 15)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5 Türlere Göre Ortalama Gişe ────────────────────────────────────────────
genre_revenue = {}
for genre in [g[0] for g in genre_counts[:12]]:
    mask = df['genres_list'].apply(lambda gl: genre in gl)
    genre_revenue[genre] = df[mask]['revenue_m'].mean()

gr_df = pd.DataFrame(list(genre_revenue.items()), columns=['Tür', 'Ort. Gişe (M$)'])
gr_df = gr_df.sort_values('Ort. Gişe (M$)', ascending=True)

fig, ax = plt.subplots(figsize=(13, 7))
colors_rev = ['#27ae60' if v >= gr_df['Ort. Gişe (M$)'].median() else '#e74c3c' for v in gr_df['Ort. Gişe (M$)']]
bars = ax.barh(gr_df['Tür'], gr_df['Ort. Gişe (M$)'], color=colors_rev, alpha=0.85)

for bar in bars:
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'${bar.get_width():.0f}M', va='center', fontsize=10)

ax.set_xlabel('Ortalama Gişe Geliri (Milyon $)', fontsize=12)
ax.set_title('💰 Türlere Göre Ortalama Gişe Geliri', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.6 Yıla Göre Film Üretimi & Ortalama Gişe ───────────────────────────────
yearly = df.groupby('release_year').agg(
    film_sayisi=('title', 'count'),
    ort_gelir=('revenue_m', 'mean'),
    ort_but=('budget_m', 'mean')
).reset_index()
yearly = yearly[yearly['release_year'].between(1980, 2016)]

fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()

ax1.fill_between(yearly['release_year'], yearly['ort_gelir'], alpha=0.3, color='#27ae60')
ax1.plot(yearly['release_year'], yearly['ort_gelir'], color='#27ae60', linewidth=2.5, label='Ort. Gişe (M$)')
ax1.plot(yearly['release_year'], yearly['ort_but'], color='#3498db', linewidth=2, linestyle='--', label='Ort. Bütçe (M$)')
ax2.bar(yearly['release_year'], yearly['film_sayisi'], alpha=0.2, color='gray', label='Film Sayısı')

ax1.set_xlabel('Yıl', fontsize=12)
ax1.set_ylabel('Milyon $', fontsize=12)
ax2.set_ylabel('Film Sayısı', fontsize=12)
ax1.set_title('📅 Yıla Göre Film Üretimi ve Ortalama Gişe (1980-2016)', fontsize=15, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.7 Bütçe vs Gişe Scatter Plot ───────────────────────────────────────────
fig = px.scatter(
    df.sample(min(800, len(df)), random_state=42),
    x='budget_m',
    y='revenue_m',
    color='success_label',
    color_discrete_map={'Hit': '#27ae60', 'Orta': '#f39c12', 'Riskli': '#e74c3c'},
    size='popularity',
    hover_name='title',
    hover_data={'release_year': True, 'vote_average': True, 'roi': ':.2f'},
    title='💰 Bütçe vs Gişe (Renk = Başarı Sınıfı, Boyut = Popülarite)',
    labels={'budget_m': 'Bütçe (Milyon $)', 'revenue_m': 'Gişe (Milyon $)', 'success_label': 'Başarı'},
    opacity=0.75,
    template='plotly_white'
)
fig.add_shape(type='line', x0=0, y0=0, x1=df['budget_m'].max(), y1=df['budget_m'].max(),
              line=dict(color='gray', dash='dash', width=1.5))
fig.add_annotation(x=280, y=250, text='Başabaş Çizgisi (Revenue=Budget)', showarrow=False, font=dict(color='gray'))
fig.show()

In [ ]:
# ── 4.8 IMDB Puanı Dağılımı ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(df['vote_average'], bins=40, color='#9b59b6', alpha=0.8, edgecolor='white')
axes[0].axvline(df['vote_average'].mean(), color='red', linestyle='--',
                label=f'Ortalama: {df["vote_average"].mean():.2f}')
axes[0].axvline(df['vote_average'].median(), color='orange', linestyle='--',
                label=f'Medyan: {df["vote_average"].median():.2f}')
axes[0].set_title('⭐ IMDB Puan Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Ortalama Puan')
axes[0].set_ylabel('Film Sayısı')
axes[0].legend()

# Puan vs Gişe box plot
df['puan_grubu'] = pd.cut(df['vote_average'], bins=[0, 5, 6, 7, 8, 10],
                           labels=['<5', '5-6', '6-7', '7-8', '>8'])
puan_rev = df.groupby('puan_grubu', observed=True)['revenue_m'].median()
colors_p = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
bars = axes[1].bar(puan_rev.index.astype(str), puan_rev.values, color=colors_p, alpha=0.85, edgecolor='white')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'${bar.get_height():.0f}M', ha='center', fontsize=10)
axes[1].set_title('💰 Puan Grubuna Göre Medyan Gişe', fontsize=14, fontweight='bold')
axes[1].set_xlabel('IMDB Puan Grubu')
axes[1].set_ylabel('Medyan Gişe (Milyon $)')

plt.suptitle('⭐ IMDB Puanı Analizi', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.9 En Başarılı Yönetmenler ───────────────────────────────────────────────
dir_stats = df.groupby('director').agg(
    film_sayisi=('title', 'count'),
    ort_roi=('roi', 'mean'),
    toplam_gelir=('revenue_m', 'sum'),
    ort_puan=('vote_average', 'mean')
).reset_index()

top_dirs = dir_stats[dir_stats['film_sayisi'] >= 3].nlargest(15, 'toplam_gelir')

fig = px.bar(
    top_dirs,
    x='toplam_gelir',
    y='director',
    orientation='h',
    color='ort_roi',
    color_continuous_scale='RdYlGn',
    hover_data={'film_sayisi': True, 'ort_puan': ':.1f', 'ort_roi': ':.2f'},
    title='🎬 En Başarılı Yönetmenler (Min 3 Film, Toplam Gişe)',
    labels={'toplam_gelir': 'Toplam Gişe (Milyon $)', 'director': 'Yönetmen', 'ort_roi': 'Ort. ROI'},
    template='plotly_white'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# ── 4.10 Runtime Dağılımı ─────────────────────────────────────────────────────
runtime_clean = df[df['runtime'].between(60, 240)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(runtime_clean['runtime'], bins=45, color='#1abc9c', alpha=0.85, edgecolor='white')
axes[0].axvline(runtime_clean['runtime'].mean(), color='red', linestyle='--',
                label=f'Ort: {runtime_clean["runtime"].mean():.0f} dk')
axes[0].set_title('⏱️ Film Süresi Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Süre (Dakika)')
axes[0].set_ylabel('Film Sayısı')
axes[0].legend()

# Runtime vs Revenue
axes[1].scatter(runtime_clean['runtime'], runtime_clean['revenue_m'],
                alpha=0.3, color='#1abc9c', s=20)
# Trend çizgisi
z = np.polyfit(runtime_clean['runtime'], runtime_clean['revenue_m'], 1)
p = np.poly1d(z)
x_line = np.linspace(60, 240, 100)
axes[1].plot(x_line, p(x_line), color='red', linewidth=2, label='Trend')
axes[1].set_title('⏱️ Film Süresi vs Gişe Geliri', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Süre (Dakika)')
axes[1].set_ylabel('Gişe Geliri (Milyon $)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.11 Popülariteye göre Gişe (Plotly) ─────────────────────────────────────
fig = px.scatter(
    df[df['popularity'] < 200].sample(min(600, len(df)), random_state=1),
    x='popularity',
    y='revenue_m',
    color='success_label',
    color_discrete_map={'Hit': '#27ae60', 'Orta': '#f39c12', 'Riskli': '#e74c3c'},
    hover_name='title',
    hover_data={'director': True, 'vote_average': True},
    trendline='ols',
    trendline_scope='overall',
    title='🌟 Popülarite vs Gişe Geliri (Trend Çizgili)',
    labels={'popularity': 'Popülarite Skoru', 'revenue_m': 'Gişe (Milyon $)', 'success_label': 'Başarı'},
    template='plotly_white'
)
fig.show()

In [ ]:
# ── 4.12 Döneme Göre Başarı Dağılımı (Stacked Bar) ────────────────────────────
decade_mask = df['decade'].isin(['1980s', '1990s', '2000s', '2010s'])
decade_success = df[decade_mask].groupby(['decade', 'success_label'], observed=True).size().unstack(fill_value=0)
decade_success_pct = decade_success.div(decade_success.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 7))
decade_success_pct.plot(kind='bar', stacked=True, ax=ax,
                         color=['#e74c3c', '#f39c12', '#27ae60'],
                         alpha=0.85, edgecolor='white')
ax.set_title('📅 Döneme Göre Başarı Dağılımı (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Dönem')
ax.set_ylabel('Yüzde (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Başarı Sınıfı', loc='upper right')
plt.tight_layout()
plt.show()

## 🔥 5. Isı Haritası — Korelasyon Analizi

In [ ]:
# ── 5.1 Korelasyon Matrisi (Sayısal Sütunlar) ────────────────────────────────
corr_cols = [
    'budget_m', 'revenue_m', 'roi', 'profit_m',
    'vote_average', 'vote_count', 'popularity',
    'runtime', 'genre_count', 'director_avg_roi', 'cast_power'
]
corr_labels = [
    'Bütçe (M$)', 'Gişe (M$)', 'ROI', 'Kâr (M$)',
    'IMDB Puanı', 'Oy Sayısı', 'Popülarite',
    'Süre', 'Tür Sayısı', 'Yönetmen ROI', 'Oyuncu Gücü'
]

corr_matrix = df[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
cmap = sns.diverging_palette(230, 20, as_cmap=True)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap=cmap,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('🔥 Korelasyon Isı Haritası — Film Özellikleri', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.2 Gişe ile En Güçlü Korelasyonlar ──────────────────────────────────────
revenue_corr = df[corr_cols].corr()['revenue'].drop('revenue').sort_values()
revenue_corr.index = [corr_labels[corr_cols.index(c)] for c in revenue_corr.index]

colors_corr = ['#e74c3c' if v < 0 else '#27ae60' for v in revenue_corr.values]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(revenue_corr.index, revenue_corr.values, color=colors_corr, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, revenue_corr.values):
    ax.text(val + (0.01 if val >= 0 else -0.02), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10,
            ha='left' if val >= 0 else 'right')
ax.set_title('🔗 Gişe Geliri ile Korelasyon Katsayıları', fontsize=14, fontweight='bold')
ax.set_xlabel('Pearson Korelasyon Katsayısı')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Başarı Sınıfına Göre Özellik Dağılımı (Violin) ───────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

violin_cols = [
    ('budget_m', 'Bütçe (M$)'),
    ('vote_average', 'IMDB Puanı'),
    ('popularity', 'Popülarite'),
    ('roi', 'ROI'),
    ('runtime', 'Süre (dk)'),
    ('director_avg_roi', 'Yönetmen Ort. ROI'),
]

palette = {'Riskli': '#e74c3c', 'Orta': '#f39c12', 'Hit': '#27ae60'}
order = ['Riskli', 'Orta', 'Hit']

for ax, (col, label) in zip(axes.flatten(), violin_cols):
    plot_df = df[df[col].between(df[col].quantile(0.02), df[col].quantile(0.98))]
    sns.violinplot(data=plot_df, x='success_label', y=col, palette=palette, order=order,
                   ax=ax, inner='quartile', alpha=0.8)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('🎻 Başarı Sınıfına Göre Özellik Dağılımları (Violin Plot)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 🤖 6. Random Forest Analizi

In [ ]:
# ── 6.1 Veri Hazırlama ────────────────────────────────────────────────────────
FEATURE_COLS = [
    'budget', 'runtime', 'popularity', 'vote_count',
    'vote_average', 'genre_count', 'director_avg_roi', 'cast_power'
]
FEATURE_LABELS = [
    'Bütçe', 'Süre', 'Popülarite', 'Oy Sayısı',
    'IMDB Puanı', 'Tür Sayısı', 'Yönetmen ROI', 'Oyuncu Gücü'
]

X = df[FEATURE_COLS].fillna(0)
y_class = df['success_class']   # 0=Riskli, 1=Orta, 2=Hit
y_rev   = df['revenue']         # Gişe tahmini

X_train, X_test, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_rev, test_size=0.2, random_state=42)

print(f'✅ Train seti: {len(X_train):,} | Test seti: {len(X_test):,}')
print(f'   Özellik sayısı: {X.shape[1]}')

In [ ]:
# ── 6.2 Random Forest Classifier Eğitimi ─────────────────────────────────────
print('🌲 Random Forest Classifier eğitiliyor...')
rf_clf = RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_c)
y_pred_c = rf_clf.predict(X_test)
rf_clf_acc = accuracy_score(y_test_c, y_pred_c)

print(f'✅ RF Classifier — Test Doğruluğu: %{rf_clf_acc*100:.2f}')
print()
print('📊 Sınıflandırma Raporu:')
print(classification_report(y_test_c, y_pred_c, target_names=['Riskli', 'Orta', 'Hit']))

In [ ]:
# ── 6.3 Confusion Matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test_c, y_pred_c)
cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sayı bazlı
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Riskli', 'Orta', 'Hit'],
            yticklabels=['Riskli', 'Orta', 'Hit'])
axes[0].set_title('Confusion Matrix (Sayı)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Gerçek Sınıf')
axes[0].set_xlabel('Tahmin Edilen Sınıf')

# Yüzde bazlı
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1],
            xticklabels=['Riskli', 'Orta', 'Hit'],
            yticklabels=['Riskli', 'Orta', 'Hit'])
axes[1].set_title('Confusion Matrix (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Gerçek Sınıf')
axes[1].set_xlabel('Tahmin Edilen Sınıf')

plt.suptitle(f'🤖 Random Forest Classifier — Doğruluk: %{rf_clf_acc*100:.2f}',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4 Feature Importance (RF Classifier) ────────────────────────────────────
fi_clf = pd.Series(rf_clf.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
palette_fi = sns.color_palette('RdYlGn', len(fi_clf))
bars = ax.barh(fi_clf.index, fi_clf.values, color=palette_fi, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, fi_clf.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('Feature Importance (Gini İmpurity)', fontsize=12)
ax.set_title('🌲 Random Forest — Özellik Önem Sıralaması (Classifier)', fontsize=14, fontweight='bold')
ax.set_xlim(0, max(fi_clf.values) * 1.15)
plt.tight_layout()
plt.show()

print('\n📊 Özellik Önem Sıralaması:')
for feat, imp in fi_clf.sort_values(ascending=False).items():
    bar_str = '█' * int(imp * 200)
    print(f'  {feat:<20}: {imp:.4f}  {bar_str}')

In [ ]:
# ── 6.5 Random Forest Regressor (Gişe Tahmini) ───────────────────────────────
print('🌲 Random Forest Regressor eğitiliyor...')
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)
y_pred_r = rf_reg.predict(X_test_r)

rf_r2 = r2_score(y_test_r, y_pred_r)
rf_mae = mean_absolute_error(y_test_r, y_pred_r) / 1e6
rf_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r)) / 1e6

print(f'✅ RF Regressor Sonuçları:')
print(f'   R² Skoru : {rf_r2:.4f}  ({rf_r2*100:.2f}% varyans açıklandı)')
print(f'   MAE       : ${rf_mae:.1f}M')
print(f'   RMSE      : ${rf_rmse:.1f}M')

In [ ]:
# ── 6.6 RF Regressor — Gerçek vs Tahmin ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Gerçek vs Tahmin
actual_m = y_test_r.values / 1e6
pred_m   = y_pred_r / 1e6

# Aşırı değerleri kırp
mask300 = (actual_m < 800) & (pred_m < 800)
a300, p300 = actual_m[mask300], pred_m[mask300]

axes[0].scatter(a300, p300, alpha=0.3, s=25, color='#3498db')
max_val = max(a300.max(), p300.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Mükemmel Tahmin')
axes[0].set_xlabel('Gerçek Gişe (Milyon $)')
axes[0].set_ylabel('Tahmin Edilen Gişe (Milyon $)')
axes[0].set_title(f'RF Regressor — Gerçek vs Tahmin\nR² = {rf_r2:.4f}', fontsize=13, fontweight='bold')
axes[0].legend()

# Residual plot
residuals = actual_m - pred_m
residuals_plot = residuals[mask300]
axes[1].scatter(p300, residuals_plot, alpha=0.3, s=25, color='#e67e22')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Tahmin Edilen Gişe (Milyon $)')
axes[1].set_ylabel('Hata (Gerçek - Tahmin, Milyon $)')
axes[1].set_title('RF Regressor — Residual (Hata) Grafiği', fontsize=13, fontweight='bold')

plt.suptitle('🌲 Random Forest Regressor — Tahmin Performansı', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 📈 7. Lineer Regresyon Analizi

In [ ]:
# ── 7.1 Basit Lineer Regresyon: Bütçe → Gişe ─────────────────────────────────
X_lr_simple = df[['budget_m']].values
y_lr_simple = df['revenue_m'].values

lr_simple = LinearRegression()
lr_simple.fit(X_lr_simple, y_lr_simple)
y_lr_simple_pred = lr_simple.predict(X_lr_simple)
r2_simple = r2_score(y_lr_simple, y_lr_simple_pred)

fig, ax = plt.subplots(figsize=(13, 7))
scatter = ax.scatter(df['budget_m'], df['revenue_m'],
                     c=df['success_class'], cmap='RdYlGn',
                     alpha=0.4, s=25)
budget_line = np.linspace(0, df['budget_m'].max(), 100)
rev_pred_line = lr_simple.coef_[0] * budget_line + lr_simple.intercept_
ax.plot(budget_line, rev_pred_line, 'r-', linewidth=2.5,
        label=f'Lineer: Gişe = {lr_simple.coef_[0]:.2f}×Bütçe + {lr_simple.intercept_:.1f}')
ax.plot([0, df['budget_m'].max()], [0, df['budget_m'].max()], 'k--', alpha=0.4, linewidth=1.5, label='Başabaş Çizgisi')
ax.set_xlabel('Bütçe (Milyon $)', fontsize=12)
ax.set_ylabel('Gişe Geliri (Milyon $)', fontsize=12)
ax.set_title(f'📈 Basit Lineer Regresyon: Bütçe → Gişe | R² = {r2_simple:.4f}', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.colorbar(scatter, ax=ax, label='Başarı Sınıfı (0=Riskli, 1=Orta, 2=Hit)')
plt.tight_layout()
plt.show()

print(f'\n📊 Basit Lineer Regresyon Sonuçları:')
print(f'   Katsayı (β₁)  : {lr_simple.coef_[0]:.4f}')
print(f'   Intercept (β₀): {lr_simple.intercept_:.2f} M$')
print(f'   R²             : {r2_simple:.4f}')

In [ ]:
# ── 7.2 Çoklu Lineer Regresyon ────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled_train = scaler.fit_transform(X_train_r)
X_scaled_test  = scaler.transform(X_test_r)

lr_multi = Ridge(alpha=1.0)
lr_multi.fit(X_scaled_train, y_train_r)
y_lr_multi_pred = lr_multi.predict(X_scaled_test)

lr_r2   = r2_score(y_test_r, y_lr_multi_pred)
lr_mae  = mean_absolute_error(y_test_r, y_lr_multi_pred) / 1e6
lr_rmse = np.sqrt(mean_squared_error(y_test_r, y_lr_multi_pred)) / 1e6

print('📊 Çoklu Lineer Regresyon (Ridge) Sonuçları:')
print(f'   R² Skoru : {lr_r2:.4f}  ({lr_r2*100:.2f}%)')
print(f'   MAE       : ${lr_mae:.1f}M')
print(f'   RMSE      : ${lr_rmse:.1f}M')

# Katsayı görselleştirme
coef_df = pd.Series(lr_multi.coef_, index=FEATURE_LABELS).sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
colors_c = ['#e74c3c' if v < 0 else '#27ae60' for v in coef_df.values]
bars = ax.barh(coef_df.index, coef_df.values, color=colors_c, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, coef_df.values):
    ax.text(val + (500000 if val >= 0 else -500000), bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.1f}M', va='center', fontsize=10,
            ha='left' if val >= 0 else 'right')
ax.set_title(f'📈 Çoklu Lineer Regresyon Katsayıları (Ridge) | R² = {lr_r2:.4f}', fontsize=14, fontweight='bold')
ax.set_xlabel('Katsayı (Standardize edilmiş özellikler üzerinden)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Lineer Regresyon — Gerçek vs Tahmin ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

actual_m_lr = y_test_r.values / 1e6
pred_m_lr   = y_lr_multi_pred / 1e6
mask_lr = (actual_m_lr < 800) & (pred_m_lr < 800)

axes[0].scatter(actual_m_lr[mask_lr], pred_m_lr[mask_lr], alpha=0.3, s=20, color='#9b59b6')
max_v = max(actual_m_lr[mask_lr].max(), pred_m_lr[mask_lr].max())
axes[0].plot([0, max_v], [0, max_v], 'r--', linewidth=2, label='Mükemmel Tahmin')
axes[0].set_xlabel('Gerçek Gişe (Milyon $)')
axes[0].set_ylabel('Tahmin (Milyon $)')
axes[0].set_title(f'Lineer Regresyon — Gerçek vs Tahmin\nR² = {lr_r2:.4f}', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals
resid_lr = actual_m_lr - pred_m_lr
axes[1].hist(resid_lr[mask_lr], bins=50, color='#9b59b6', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Hata (Gerçek - Tahmin, Milyon $)')
axes[1].set_ylabel('Frekans')
axes[1].set_title('Lineer Regresyon — Hata Dağılımı', fontsize=13, fontweight='bold')

plt.suptitle('📈 Lineer Regresyon Performansı', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 8. Model Doğruluk Metrikleri — Karşılaştırma

In [ ]:
# ── 8.1 GradientBoosting Regressor Eğitimi ────────────────────────────────────
from sklearn.ensemble import GradientBoostingRegressor

print('⚙️  GradientBoostingRegressor eğitiliyor...')
gbr = GradientBoostingRegressor(n_estimators=250, learning_rate=0.08,
                                  max_depth=5, subsample=0.8, random_state=42)
gbr.fit(X_train_r, y_train_r)
y_gbr_pred = gbr.predict(X_test_r)

gbr_r2   = r2_score(y_test_r, y_gbr_pred)
gbr_mae  = mean_absolute_error(y_test_r, y_gbr_pred) / 1e6
gbr_rmse = np.sqrt(mean_squared_error(y_test_r, y_gbr_pred)) / 1e6

print(f'✅ GBR — R²: {gbr_r2:.4f} | MAE: ${gbr_mae:.1f}M | RMSE: ${gbr_rmse:.1f}M')

# Ana model olarak kaydet
reg_model = gbr
clf_model = rf_clf
print('\n✅ Ana modeller kaydedildi.')

In [ ]:
# ── 8.2 Model Karşılaştırma Tablosu ──────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['Lineer Regresyon (Ridge)', 'Random Forest Reg.', 'Gradient Boosting Reg.'],
    'R² Skoru': [lr_r2, rf_r2, gbr_r2],
    'MAE (M$)': [lr_mae, rf_mae, gbr_mae],
    'RMSE (M$)': [lr_rmse, rf_rmse, gbr_rmse],
})
comparison = comparison.sort_values('R² Skoru', ascending=False)

print('='*65)
print('  MODEL KARŞILAŞTIRMA (Gişe Tahmini — Regresyon)')
print('='*65)
print(comparison.to_string(index=False, float_format='{:.4f}'.format))
print('='*65)

In [ ]:
# ── 8.3 Model Karşılaştırma Grafiği ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

models = comparison['Model'].tolist()
colors_m = ['#e74c3c', '#3498db', '#27ae60'][:len(models)]

# R²
axes[0].bar(models, comparison['R² Skoru'], color=colors_m, alpha=0.85, edgecolor='white')
for i, v in enumerate(comparison['R² Skoru']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('R² Skoru (↑ İyi)', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.05)
axes[0].set_xticklabels(models, rotation=15, ha='right')

# MAE
axes[1].bar(models, comparison['MAE (M$)'], color=colors_m, alpha=0.85, edgecolor='white')
for i, v in enumerate(comparison['MAE (M$)']):
    axes[1].text(i, v + 0.5, f'${v:.1f}M', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('MAE — Ort. Mutlak Hata (↓ İyi)', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(models, rotation=15, ha='right')

# RMSE
axes[2].bar(models, comparison['RMSE (M$)'], color=colors_m, alpha=0.85, edgecolor='white')
for i, v in enumerate(comparison['RMSE (M$)']):
    axes[2].text(i, v + 0.5, f'${v:.1f}M', ha='center', fontsize=11, fontweight='bold')
axes[2].set_title('RMSE — Kök Ort. Kare Hata (↓ İyi)', fontsize=13, fontweight='bold')
axes[2].set_xticklabels(models, rotation=15, ha='right')

plt.suptitle('🎯 Regresyon Modelleri Performans Karşılaştırması', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.4 Classifier Çapraz Doğrulama ──────────────────────────────────────────
print('🔄 5-Fold Çapraz Doğrulama (Random Forest Classifier)...')
cv_scores = cross_val_score(rf_clf, X, y_class, cv=5, scoring='accuracy', n_jobs=-1)

print(f'\n📊 5-Fold Cross-Validation Sonuçları:')
print(f'   Fold Skorları: {[f"%{s*100:.2f}" for s in cv_scores]}')
print(f'   Ortalama     : %{cv_scores.mean()*100:.2f}')
print(f'   Std Sapma    : %{cv_scores.std()*100:.2f}')
print(f'   Min-Max      : %{cv_scores.min()*100:.2f} — %{cv_scores.max()*100:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars = ax.bar(folds, cv_scores * 100, color='#3498db', alpha=0.85, edgecolor='white')
ax.axhline(cv_scores.mean() * 100, color='red', linestyle='--', linewidth=2,
           label=f'Ort: %{cv_scores.mean()*100:.2f}')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'%{bar.get_height():.2f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_title('🔄 5-Fold Cross-Validation — RF Classifier', fontsize=14, fontweight='bold')
ax.set_ylabel('Doğruluk (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.5 GBR Feature Importance ────────────────────────────────────────────────
fi_gbr = pd.Series(gbr.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# GBR
palette_gbr = sns.color_palette('PiYG', len(fi_gbr))
axes[0].barh(fi_gbr.index, fi_gbr.values, color=palette_gbr, alpha=0.85, edgecolor='white')
for i, (idx, val) in enumerate(fi_gbr.items()):
    axes[0].text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=10)
axes[0].set_title('Gradient Boosting Reg.', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature Importance')

# RF Classifier
fi_rfc = pd.Series(rf_clf.feature_importances_, index=FEATURE_LABELS).sort_values(ascending=True)
palette_rfc = sns.color_palette('coolwarm', len(fi_rfc))
axes[1].barh(fi_rfc.index, fi_rfc.values, color=palette_rfc, alpha=0.85, edgecolor='white')
for i, (idx, val) in enumerate(fi_rfc.items()):
    axes[1].text(val + 0.002, i, f'{val:.4f}', va='center', fontsize=10)
axes[1].set_title('Random Forest Classifier', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Feature Importance')

plt.suptitle('🌲 Model Özellik Önemi Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.6 Özet Metrikler Paneli ────────────────────────────────────────────────
print('\n' + '='*65)
print('  📊 ÖZET MODEL METRİKLERİ')
print('='*65)
print(f'\n  🌲 Random Forest Classifier (Başarı Sınıfı Tahmini)')
print(f'     • Test Doğruluğu   : %{rf_clf_acc*100:.2f}')
print(f'     • CV Ort. Doğruluk : %{cv_scores.mean()*100:.2f} ± %{cv_scores.std()*100:.2f}')
print(f'     • Sınıflar         : Riskli (ROI<1) / Orta (ROI 1–2) / Hit (ROI>2)')
print()
print(f'  📈 Gradient Boosting Regressor (Gişe Tahmini)')
print(f'     • R² Skoru         : {gbr_r2:.4f}  ({gbr_r2*100:.2f}%)')
print(f'     • MAE              : ${gbr_mae:.1f}M')
print(f'     • RMSE             : ${gbr_rmse:.1f}M')
print()
print(f'  📊 Random Forest Regressor (Karşılaştırma)')
print(f'     • R² Skoru         : {rf_r2:.4f}  ({rf_r2*100:.2f}%)')
print(f'     • MAE              : ${rf_mae:.1f}M')
print(f'     • RMSE             : ${rf_rmse:.1f}M')
print()
print(f'  📉 Lineer Regresyon — Ridge (Karşılaştırma)')
print(f'     • R² Skoru         : {lr_r2:.4f}  ({lr_r2*100:.2f}%)')
print(f'     • MAE              : ${lr_mae:.1f}M')
print(f'     • RMSE             : ${lr_rmse:.1f}M')
print('='*65)

## 🎬 9. Film Test Et — İstediğin Filmi Tahmin Et!

Aşağıdaki bölümde **iki farklı yöntemle** film test edebilirsiniz:

- **9A** → Dataset'te mevcut bir filmi ada göre ara
- **9B** → Manuel parametreler girerek yeni/olmayan bir filmi tahmin et

In [ ]:
# ── 9A. Dataset'te Arama ve Analiz ────────────────────────────────────────────
# 🎯 BU HÜCREYİ DEĞİŞTİR: İstediğin film adını yaz!

ARAMA_FILMI = "Inception"  # ← Buraya film adını yaz (İngilizce)

# ─────────────────────────────────────────────────────────────────────────────
results = df[df['title'].str.contains(ARAMA_FILMI, case=False, na=False)]

if results.empty:
    print(f'❌ "{ARAMA_FILMI}" dataset\'te bulunamadı.')
    print('💡 İpucu: İngilizce adı tam yazmayı dene veya kısmi ad kullan.')
    print('\n🔍 Benzer filmler aranıyor...')
    words = ARAMA_FILMI.split()
    for w in words:
        partial = df[df['title'].str.contains(w, case=False, na=False)]['title'].head(5).tolist()
        if partial:
            print(f'  "{w}" içeren filmler: {partial}')
else:
    for _, film in results.iterrows():
        print('='*60)
        print(f"  🎬 {film['title']}")
        print('='*60)
        print(f"  📅 Yıl       : {str(film.get('release_date',''))[:4]}")
        print(f"  🎭 Türler    : {', '.join(film['genres_list']) if film['genres_list'] else '—'}")
        print(f"  🎬 Yönetmen  : {film.get('director', '—')}")
        print(f"  ⭐ IMDB      : {film['vote_average']:.1f}/10  ({film['vote_count']:,} oy)")
        print(f"  🌟 Popülarite: {film['popularity']:.1f}")
        print(f"  💰 Bütçe     : ${film['budget_m']:.1f}M")
        print(f"  💵 Gişe      : ${film['revenue_m']:.1f}M")
        print(f"  📈 ROI       : x{film['roi']:.2f}")
        print(f"  💼 Kâr       : ${film['profit_m']:.1f}M")
        print(f"  🏆 Sonuç     : {film['success_label']}")
        print(f"  ⏱️  Süre      : {film.get('runtime', '?')} dk")
        overview = str(film.get('overview', ''))[:200]
        if overview:
            print(f"\n  📖 Özet: {overview}...")
        print()

In [ ]:
# ── 9A.2 Dataset'teki film için model tahmini ─────────────────────────────────
# (film dataset'te mevcut olduğunda çalışır)

if not results.empty:
    film_row = results.iloc[0]
    film_X = film_row[FEATURE_COLS].fillna(0).values.reshape(1, -1)

    pred_rev   = reg_model.predict(film_X)[0]
    pred_proba = clf_model.predict_proba(film_X)[0]

    prob_fail = pred_proba[0] * 100
    prob_mid  = pred_proba[1] * 100
    prob_hit  = pred_proba[2] * 100 if len(pred_proba) == 3 else (100 - prob_fail - prob_mid)

    pred_roi    = pred_rev / max(film_row['budget'], 1)
    actual_rev  = film_row['revenue_m']
    hata_pct    = abs(pred_rev/1e6 - actual_rev) / actual_rev * 100 if actual_rev > 0 else None

    print(f"🤖 MODEL TAHMİNLERİ — {film_row['title']}")
    print('─'*50)
    print(f"  Tahmini Gişe : ${pred_rev/1e6:.1f}M")
    print(f"  Gerçek Gişe  : ${actual_rev:.1f}M")
    if hata_pct:
        print(f"  Hata         : %{hata_pct:.1f}")
    print(f"  Tahmini ROI  : x{pred_roi:.2f}")
    print()
    print(f"  🔴 Riskli Olasılığı : %{prob_fail:.1f}")
    print(f"  🟡 Orta Olasılığı   : %{prob_mid:.1f}")
    print(f"  🟢 Hit Olasılığı    : %{prob_hit:.1f}")

In [ ]:
# ── 9A.3 Dataset'teki film görsel analizi ─────────────────────────────────────
if not results.empty:
    film_row = results.iloc[0]
    film_title = film_row['title']
    film_genres = film_row['genres_list']

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'bar'}, {'type': 'pie'}]],
        subplot_titles=[
            f'Başarı Olasılıkları — {film_title}',
            f'Tür Başarı Karşılaştırması'
        ]
    )

    # Olasılık bar chart
    fig.add_trace(
        go.Bar(
            x=['🔴 Riskli', '🟡 Orta', '🟢 Hit'],
            y=[prob_fail, prob_mid, prob_hit],
            marker_color=['#e74c3c', '#f39c12', '#27ae60'],
            text=[f'%{prob_fail:.1f}', f'%{prob_mid:.1f}', f'%{prob_hit:.1f}'],
            textposition='outside',
            showlegend=False
        ), row=1, col=1
    )

    # Tür başarı oranı
    genre_hit_rates = {}
    for g in film_genres[:5]:
        mask = df['genres_list'].apply(lambda gl: g in gl)
        genre_hit_rates[g] = (df[mask]['success_class'] == 2).mean() * 100

    if genre_hit_rates:
        fig.add_trace(
            go.Pie(
                labels=list(genre_hit_rates.keys()),
                values=list(genre_hit_rates.values()),
                showlegend=True
            ), row=1, col=2
        )

    fig.update_layout(
        title_text=f'🎬 {film_title} — Model Analiz Paneli',
        title_font_size=16,
        template='plotly_white',
        height=450
    )
    fig.update_yaxes(range=[0, 115], row=1, col=1)
    fig.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ── 9B. MANUEL FİLM TAHMİNİ ──────────────────────────────────────────────────
# 🎯 Aşağıdaki değerleri değiştirerek istediğin filmi test et!
# ══════════════════════════════════════════════════════════════════════════════

MANUEL_FILM = {
    # ── Temel bilgiler ────────────────────────────────────────
    'title'        : 'Avatar: The Way of Water',  # Film adı
    'budget'       : 350_000_000,                  # Bütçe ($) — 0 ise tür bazlı tahmin yapılır
    'runtime'      : 192,                          # Süre (dakika)
    'popularity'   : 130.0,                        # Popülarite skoru
    'vote_count'   : 9000,                         # Oy sayısı
    'vote_average' : 7.6,                          # IMDB puanı (1-10)

    # ── Tür bilgisi ───────────────────────────────────────────
    # Seçenekler: Action, Adventure, Animation, Comedy, Crime,
    #             Drama, Fantasy, Horror, Romance, Science Fiction,
    #             Thriller, Mystery, Family, History, Music
    'genres'       : ['Action', 'Adventure', 'Science Fiction'],

    # ── Yönetmen ve oyuncular ────────────────────────────────
    'directors'    : ['James Cameron'],
    'cast'         : ['Sam Worthington', 'Zoe Saldana', 'Sigourney Weaver'],
}

# ─────────────────────────────────────────────────────────────────────────────
# Hesaplamalar
budget     = MANUEL_FILM['budget']
runtime    = MANUEL_FILM['runtime'] or 100
popularity = MANUEL_FILM['popularity']
vote_count = MANUEL_FILM['vote_count']
vote_avg   = MANUEL_FILM['vote_average']
genres     = MANUEL_FILM['genres']
directors  = MANUEL_FILM['directors']
cast       = MANUEL_FILM['cast']
genre_count = len(genres)

# Yönetmen ROI
director    = directors[0] if directors else 'Unknown'
dir_data    = df[df['director'] == director]
dir_roi_med = df['director_avg_roi'].median()
dir_roi     = dir_data['roi'].mean() if len(dir_data) > 0 else dir_roi_med
dir_roi     = dir_roi if not np.isnan(dir_roi) else dir_roi_med

# Oyuncu gücü
cast_rev_map = df.explode('cast_list').groupby('cast_list')['revenue'].mean()
cast_values  = [cast_rev_map.get(c, 0) for c in cast]
cast_power   = np.mean(cast_values) if any(v > 0 for v in cast_values) else df['cast_power'].median()

# Bütçe tahmini (bütçe girilmemişse)
GENRE_BUDGET = {
    'Action': 180e6, 'Adventure': 150e6, 'Animation': 120e6,
    'Comedy': 60e6, 'Drama': 40e6, 'Horror': 25e6,
    'Science Fiction': 140e6, 'Thriller': 55e6,
    'Romance': 35e6, 'Fantasy': 120e6, 'Crime': 50e6,
}
budget_estimated = budget <= 0
if budget_estimated:
    budget = GENRE_BUDGET.get(genres[0] if genres else 'Drama', 70e6)
    print(f'⚠️  Bütçe girilmedi → Tür bazlı tahmini bütçe: ${budget/1e6:.0f}M')

# Model tahmini
X_input = [[budget, runtime, popularity, vote_count, vote_avg, genre_count, dir_roi, cast_power]]
pred_rev   = reg_model.predict(X_input)[0]
pred_proba = clf_model.predict_proba(X_input)[0]

prob_fail = pred_proba[0] * 100
prob_mid  = pred_proba[1] * 100
prob_hit  = pred_proba[2] * 100 if len(pred_proba) == 3 else (100 - prob_fail - prob_mid)

pred_roi    = pred_rev / budget
pred_profit = pred_rev - budget

# Tür hit oranı
genre_hit_rates = {}
for g in genres:
    mask_g = df['genres_list'].apply(lambda gl: g in gl)
    genre_hit_rates[g] = (df[mask_g]['success_class'] == 2).mean() * 100
avg_genre_hit = np.mean(list(genre_hit_rates.values())) if genre_hit_rates else 30

# Verdict
if pred_roi >= 2.0:
    verdict = '🟢 HIT'
elif pred_roi >= 1.0:
    verdict = '🟡 ORTA'
else:
    verdict = '🔴 RİSKLİ'

print(f"\n{'='*60}")
print(f"  🎬 {MANUEL_FILM['title']} — TAHMİN RAPORU")
print(f"{'='*60}")
print(f"  💰 Bütçe           : ${budget/1e6:.1f}M{'  (tahmini)' if budget_estimated else ''}")
print(f"  ⏱️  Süre             : {runtime} dk")
print(f"  🌟 Popülarite      : {popularity:.1f}")
print(f"  ⭐ IMDB Puanı      : {vote_avg}/10")
print(f"  🎭 Türler          : {', '.join(genres)}")
print(f"  🎬 Yönetmen        : {director} (Ort. ROI x{dir_roi:.2f})")
print(f"  🦸 Oyuncular       : {', '.join(cast)}")
print()
print(f"  📈 Tahmini Gişe    : ${pred_rev/1e6:.1f}M")
print(f"  📈 Tahmini ROI     : x{pred_roi:.2f}")
print(f"  💼 Tahmini Kâr     : ${pred_profit/1e6:.1f}M")
print()
print(f"  🔴 Riskli Olasılığı: %{prob_fail:.1f}")
print(f"  🟡 Orta Olasılığı  : %{prob_mid:.1f}")
print(f"  🟢 Hit Olasılığı   : %{prob_hit:.1f}")
print()
print(f"  🏆 KARAR           : {verdict}")
print(f"{'='*60}")

In [ ]:
# ── 9B.2 Manuel Film — Görsel Analiz ──────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    specs=[
        [{'type': 'indicator'}, {'type': 'bar'}],
        [{'type': 'bar'}, {'type': 'scatter'}]
    ],
    subplot_titles=[
        'Tahmini ROI',
        'Başarı Olasılıkları',
        'Tür Hit Oranları',
        'Bütçe vs Tahmini Gişe Konumu'
    ]
)

# ROI gauge
fig.add_trace(
    go.Indicator(
        mode='gauge+number+delta',
        value=pred_roi,
        delta={'reference': 2.0},
        title={'text': 'ROI (Hedef: 2.0)'},
        gauge={
            'axis': {'range': [0, 5]},
            'bar': {'color': '#27ae60' if pred_roi >= 2 else ('#f39c12' if pred_roi >= 1 else '#e74c3c')},
            'steps': [
                {'range': [0, 1], 'color': '#fee'},
                {'range': [1, 2], 'color': '#ffe'},
                {'range': [2, 5], 'color': '#efe'},
            ],
            'threshold': {'line': {'color': 'red', 'width': 2}, 'thickness': 0.75, 'value': 2}
        }
    ), row=1, col=1
)

# Olasılık bar chart
fig.add_trace(
    go.Bar(
        x=['Riskli', 'Orta', 'Hit'],
        y=[prob_fail, prob_mid, prob_hit],
        marker_color=['#e74c3c', '#f39c12', '#27ae60'],
        text=[f'%{prob_fail:.1f}', f'%{prob_mid:.1f}', f'%{prob_hit:.1f}'],
        textposition='outside',
        showlegend=False
    ), row=1, col=2
)

# Tür hit oranları
if genre_hit_rates:
    fig.add_trace(
        go.Bar(
            x=list(genre_hit_rates.keys()),
            y=list(genre_hit_rates.values()),
            marker_color='#3498db',
            text=[f'%{v:.1f}' for v in genre_hit_rates.values()],
            textposition='outside',
            showlegend=False
        ), row=2, col=1
    )

# Bütçe vs gişe scatter (dataset arka plan)
sample_df = df.sample(min(500, len(df)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample_df['budget_m'], y=sample_df['revenue_m'],
        mode='markers',
        marker=dict(color='lightgray', size=5, opacity=0.5),
        name='Dataset Filmleri',
        showlegend=True
    ), row=2, col=2
)
fig.add_trace(
    go.Scatter(
        x=[budget/1e6], y=[pred_rev/1e6],
        mode='markers+text',
        marker=dict(color='#e74c3c', size=20, symbol='star'),
        text=[MANUEL_FILM['title']],
        textposition='top center',
        name=MANUEL_FILM['title'],
        showlegend=True
    ), row=2, col=2
)

fig.update_layout(
    title_text=f"🎬 {MANUEL_FILM['title']} — Kapsamlı Tahmin Paneli",
    title_font_size=16,
    template='plotly_white',
    height=700
)
fig.update_yaxes(range=[0, 120], row=1, col=2)
fig.update_yaxes(range=[0, max(list(genre_hit_rates.values())) * 1.2 if genre_hit_rates else 100], row=2, col=1)
fig.show()

In [ ]:
# ── 9C. Toplu Film Karşılaştırma ──────────────────────────────────────────────
# Birden fazla filmi karşılaştır!

FILMLER = [
    {'title': 'Düşük Bütçeli Korku',  'budget': 15e6,  'runtime': 90,  'popularity': 20, 'vote_count': 500, 'vote_average': 6.5, 'genres': ['Horror'], 'directors': ['Unknown'], 'cast': []},
    {'title': 'Orta Bütçeli Dram',    'budget': 50e6,  'runtime': 120, 'popularity': 45, 'vote_count': 2000, 'vote_average': 7.5, 'genres': ['Drama'], 'directors': ['Unknown'], 'cast': []},
    {'title': 'Büyük Bütçeli Aksiyon','budget': 200e6, 'runtime': 150, 'popularity': 100,'vote_count': 8000, 'vote_average': 7.8, 'genres': ['Action', 'Adventure'], 'directors': ['Unknown'], 'cast': []},
    {'title': 'Sci-Fi Blockbuster',   'budget': 300e6, 'runtime': 170, 'popularity': 130,'vote_count': 12000,'vote_average': 8.2, 'genres': ['Science Fiction', 'Action'], 'directors': ['James Cameron'], 'cast': ['Sam Worthington']},
]

# ─────────────────────────────────────────────────────────────────────────────
results_bulk = []

for film in FILMLER:
    b  = film['budget']
    rt = film['runtime']
    po = film['popularity']
    vc = film['vote_count']
    va = film['vote_average']
    gc = len(film['genres'])

    d_name = film['directors'][0] if film['directors'] else 'Unknown'
    d_data = df[df['director'] == d_name]
    d_roi  = d_data['roi'].mean() if len(d_data) > 0 else df['director_avg_roi'].median()

    c_vals = [cast_rev_map.get(c, 0) for c in film['cast']]
    cp     = np.mean(c_vals) if any(v > 0 for v in c_vals) else df['cast_power'].median()

    Xi = [[b, rt, po, vc, va, gc, d_roi, cp]]
    pr  = reg_model.predict(Xi)[0]
    pp  = clf_model.predict_proba(Xi)[0]

    results_bulk.append({
        'Film'        : film['title'],
        'Bütçe (M$)'  : b / 1e6,
        'Tahmini Gişe': pr / 1e6,
        'Tahmini ROI' : pr / b,
        'Tahmini Kâr' : (pr - b) / 1e6,
        'Hit %'       : pp[2] * 100 if len(pp) == 3 else 0,
        'Riskli %'    : pp[0] * 100,
    })

df_bulk = pd.DataFrame(results_bulk)

print('📊 TOPLU FİLM KARŞILAŞTIRMASI')
print(df_bulk.round(1).to_string(index=False))

# Görsel
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x_pos = range(len(df_bulk))
width = 0.35

axes[0].bar([x - width/2 for x in x_pos], df_bulk['Bütçe (M$)'], width=width, label='Bütçe', color='#3498db', alpha=0.85)
axes[0].bar([x + width/2 for x in x_pos], df_bulk['Tahmini Gişe'], width=width, label='Tahmini Gişe', color='#27ae60', alpha=0.85)
axes[0].set_xticks(list(x_pos))
axes[0].set_xticklabels(df_bulk['Film'], rotation=12, ha='right')
axes[0].set_ylabel('Milyon $')
axes[0].set_title('💰 Bütçe vs Tahmini Gişe', fontsize=13, fontweight='bold')
axes[0].legend()

colors_bulk = ['#27ae60' if r >= 2 else ('#f39c12' if r >= 1 else '#e74c3c') for r in df_bulk['Tahmini ROI']]
axes[1].bar(df_bulk['Film'], df_bulk['Tahmini ROI'], color=colors_bulk, alpha=0.85)
axes[1].axhline(1, color='gray', linestyle='--', linewidth=1.5, label='Başabaş (ROI=1)')
axes[1].axhline(2, color='green', linestyle='--', linewidth=1.5, label='Hit Sınırı (ROI=2)')
for i, v in enumerate(df_bulk['Tahmini ROI']):
    axes[1].text(i, v + 0.05, f'x{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_xticks(list(x_pos))
axes[1].set_xticklabels(df_bulk['Film'], rotation=12, ha='right')
axes[1].set_ylabel('ROI')
axes[1].set_title('📈 Tahmini ROI Karşılaştırması', fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle('🎬 Çoklu Film Tahmin Karşılaştırması', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 📌 Hızlı Referans

| Hücre | İşlev |
|-------|-------|
| `ARAMA_FILMI = "..."` | Dataset'te film ara ve analiz et |
| `MANUEL_FILM = {...}` | Kendi parametrelerinle film tahmin et |
| `FILMLER = [...]` | Birden fazla filmi karşılaştır |

### Kullanılabilir Türler
`Action` · `Adventure` · `Animation` · `Comedy` · `Crime` · `Drama` · `Fantasy` · `Horror` · `Music` · `Mystery` · `Romance` · `Science Fiction` · `Thriller` · `Family` · `History`

### Başarı Sınıfları
- 🔴 **Riskli**: ROI < 1 (zarar)
- 🟡 **Orta**: ROI 1–2 (kârlı ama sıradan)
- 🟢 **Hit**: ROI > 2 (büyük başarı)